# YaadDost: Gemma server on Colab
Runs an open-weight **Gemma** model with Ollama on Colab's free GPU and exposes it through a temporary public URL,
so the YaadDost app (on Render, or on your laptop) can use it. No local disk space needed.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells top to bottom.

Heads-up: the URL printed at the end is public to anyone who has it. It is random and dies with the session, but don't share it.

In [ ]:
%%bash
apt-get install -y -qq zstd > /dev/null
curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, urllib.request

subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=1)
        break
    except Exception:
        time.sleep(1)
print("ollama is up")

In [ ]:
# gemma4:e4b is the default. Use gemma4:e2b for a smaller/faster model (then set GEMMA_MODEL=gemma4:e2b in the app).
MODEL = "gemma4:e4b"
!ollama pull {MODEL}

In [ ]:
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

In [ ]:
import re, subprocess, time

log = open("/content/tunnel.log", "w")
subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:11434", "--http-host-header", "localhost:11434"],
    stdout=log, stderr=subprocess.STDOUT,
)
url = None
for _ in range(40):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break
    time.sleep(1)
print("OLLAMA_URL =", url)

In [ ]:
# Sanity check through the public URL: this should print a short reply from Gemma.
import json, urllib.request
req = urllib.request.Request(
    url + "/api/chat",
    data=json.dumps({"model": MODEL, "stream": False, "messages": [{"role": "user", "content": "Say hi in Hinglish, one line."}]}).encode(),
    headers={"Content-Type": "application/json"},
)
print(json.load(urllib.request.urlopen(req, timeout=300))["message"]["content"])

## Connect the app
Set `OLLAMA_URL` to the URL printed above:
- **Render:** Dashboard > your service > Environment > `OLLAMA_URL`, then redeploy.
- **Locally:** `set OLLAMA_URL=https://....trycloudflare.com` (PowerShell: `$env:OLLAMA_URL="..."`), then `uvicorn app.main:app`.

Keep this tab open while people use the app. When the session ends the app drops back to its offline mode.